# 🔍 AIOps Log Clustering — Pipeline CompletCe notebook implémente le pipeline complet de **log clustering pour AIOps** en s'appuyant sur les logs réels collectés depuis **Loki**.**Prérequis :** Exécuter `collect_logs.py` pour constituer le dataset CSV depuis Loki.**Référence :** Basé sur le notebook `demo-notebook-aiOps.ipynb` dans `.agent/skills/aiOps/ressources/`

## 1. Chargement du dataset CSV (collecté depuis Loki)

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport reimport warningswarnings.filterwarnings('ignore')# === Charger le dataset collecté depuis Loki ===CSV_PATH = 'data/logs_dataset.csv'try:    df_logs = pd.read_csv(CSV_PATH)    print(f"✅ Dataset chargé: {len(df_logs)} logs")    print(f"   Colonnes: {list(df_logs.columns)}")    print(f"   Période: {df_logs['timestamp'].min()} → {df_logs['timestamp'].max()}")    print(f"\n📊 Distribution par niveau:")    print(df_logs['level'].value_counts().to_string())    print(f"\n📊 Distribution par pattern:")    print(df_logs['pattern'].value_counts().to_string())except FileNotFoundError:    print("❌ Dataset non trouvé!")    print("   Exécutez d'abord: python collect_logs.py --loki-url http://<IP_LOKI>:3100")    print("   Ou en mode continu: python collect_logs.py --continuous --interval 60")

### 1.1 — Aperçu des données brutes

In [ ]:
# Visualiser les premières lignesprint("📝 Aperçu des logs bruts:")print("=" * 80)for i, row in df_logs.head(10).iterrows():    print(f"[{row['level']:>7}] {row['timestamp']} | {row['message'][:80]}")print("=" * 80)# Stats globalesprint(f"\n📊 Statistiques du dataset:")print(f"   Total logs: {len(df_logs):,}")print(f"   Patterns uniques: {df_logs['pattern'].nunique()}")print(f"   Sources: {df_logs['source'].unique()}")

### 1.2 — Rafraîchir le dataset (re-collecter depuis Loki)

In [ ]:
# Pour récupérer les derniers logs sans quitter le notebook:# Décommentez et adaptez l'URL Loki# import subprocess# result = subprocess.run(#     ['python3', 'collect_logs.py', '--loki-url', 'http://10.0.1.XX:3100', '--hours', '1'],#     capture_output=True, text=True# )# print(result.stdout)# # # Recharger le CSV mis à jour# df_logs = pd.read_csv(CSV_PATH)# print(f"\n✅ Dataset rechargé: {len(df_logs)} logs")

---## 2. Nettoyage et préparation des données

In [ ]:
# Nettoyage du texte pour le clusteringdf_logs['message_clean'] = df_logs['message'].astype(str).str.strip().str.lower()# Normaliser les valeurs variables (IPs, timestamps, IDs)def normalize_message(msg):    """Remplace les valeurs variables par des placeholders pour le clustering."""    msg = re.sub(r'\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}', '<TIMESTAMP>', msg)    msg = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '<IP>', msg)    msg = re.sub(r'\b\d+\.\d+s\b', '<DURATION>', msg)    msg = re.sub(r'\b\d+ms\b', '<DURATION>', msg)    msg = re.sub(r'\b\d+%\b', '<PERCENT>', msg)    msg = re.sub(r'port\s+\d+', 'port <PORT>', msg)    msg = re.sub(r'user_id=\d+', 'user_id=<ID>', msg)    msg = re.sub(r'session_\w+', 'session_<ID>', msg)    msg = re.sub(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b', '<EMAIL>', msg)    return msgdf_logs['message_normalized'] = df_logs['message_clean'].apply(normalize_message)print("✅ Messages normalisés")print("\nExemple avant/après:")for i in range(min(5, len(df_logs))):    print(f"  Avant:  {df_logs['message_clean'].iloc[i][:80]}")    print(f"  Après:  {df_logs['message_normalized'].iloc[i][:80]}")    print()print(f"Messages uniques (bruts):       {df_logs['message_clean'].nunique()}")print(f"Messages uniques (normalisés):  {df_logs['message_normalized'].nunique()}")reduction = (1 - df_logs['message_normalized'].nunique() / max(1, df_logs['message_clean'].nunique())) * 100print(f"Réduction:                      {reduction:.1f}%")

---## 3. TF-IDF — Vectorisation des logsConvertir les messages texte en vecteurs numériques pour que le ML puisse mesurer la similarité.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer# Combiner le pattern (déjà extrait par collect_logs) + le message normalisé# pour un clustering plus richedf_logs['clustering_text'] = df_logs['level'].str.lower() + ' ' + \                              df_logs['pattern'] + ' ' + \                              df_logs['message_normalized']# Vectorisation TF-IDFvectorizer = TfidfVectorizer(    max_features=150,    stop_words='english',    min_df=2,    max_df=0.95,    token_pattern=r'(?u)\b\w+\b')X_tfidf = vectorizer.fit_transform(df_logs['clustering_text'])print(f"✅ Vectorisation TF-IDF terminée")print(f"   Matrice: {X_tfidf.shape} (logs × features)")print(f"   Features: {X_tfidf.shape[1]} termes uniques")print(f"   Sparsité: {(1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])) * 100:.1f}%")# Top termes par importancefeature_names = vectorizer.get_feature_names_out()word_importance = np.asarray(X_tfidf.sum(axis=0)).flatten()sorted_indices = np.argsort(word_importance)[::-1]print(f"\n🔤 Top 15 termes les plus importants:")for i in range(min(15, len(feature_names))):    idx = sorted_indices[i]    print(f"   {i+1:2d}. {feature_names[idx]:<25} (poids: {word_importance[idx]:.3f})")

### 3.1 — Visualisation du vocabulaire

In [ ]:
# Graphique des top termestop_n = min(20, len(feature_names))top_words = feature_names[sorted_indices[:top_n]]top_values = word_importance[sorted_indices[:top_n]]plt.figure(figsize=(10, 6))plt.barh(top_words[::-1], top_values[::-1], color='#2196F3', alpha=0.8)plt.title('Top termes TF-IDF dans les logs', fontsize=14, fontweight='bold')plt.xlabel('Poids TF-IDF total')plt.grid(axis='x', alpha=0.3)plt.tight_layout()plt.show()

---## 4. Choix du nombre optimal de clusters (Elbow + Silhouette)

In [ ]:
from sklearn.cluster import KMeansfrom sklearn.metrics import silhouette_scoreprint("🔍 Évaluation des différents nombres de clusters...")k_values = range(2, min(12, len(df_logs)))inertias = []silhouettes = []for k in k_values:    model = KMeans(n_clusters=k, random_state=42, n_init=10)    preds = model.fit_predict(X_tfidf)    inertias.append(model.inertia_)    sil = silhouette_score(X_tfidf, preds)    silhouettes.append(sil)    print(f"   k={k:2d}: inertia={model.inertia_:10.0f}, silhouette={sil:.3f}")# Trouver le meilleur kbest_k = list(k_values)[np.argmax(silhouettes)]print(f"\n✅ Meilleur k recommandé: {best_k} (silhouette: {max(silhouettes):.3f})")

In [ ]:
# Graphiques Elbow + Silhouettefig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))ax1.plot(list(k_values), inertias, 'o-', color='#1976D2', linewidth=2, markersize=8)ax1.set_title('Méthode du coude (Elbow)', fontsize=13, fontweight='bold')ax1.set_xlabel('Nombre de clusters (k)')ax1.set_ylabel('Inertia')ax1.grid(True, alpha=0.3)ax2.plot(list(k_values), silhouettes, 's-', color='#FF5722', linewidth=2, markersize=8)ax2.axvline(x=best_k, color='green', linestyle='--', alpha=0.7, label=f'Meilleur k={best_k}')ax2.set_title('Score Silhouette', fontsize=13, fontweight='bold')ax2.set_xlabel('Nombre de clusters (k)')ax2.set_ylabel('Silhouette Score')ax2.legend()ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()

---## 5. Clustering K-Means final

In [ ]:
# Clustering avec le k optimalkmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)df_logs['cluster_id'] = kmeans.fit_predict(X_tfidf)# Distribution des clustersprint(f"✅ Clustering terminé avec k={best_k}")print(f"\n📊 Distribution des clusters:")print("-" * 50)for cid in sorted(df_logs['cluster_id'].unique()):    count = (df_logs['cluster_id'] == cid).sum()    pct = count / len(df_logs) * 100    print(f"   Cluster {cid}: {count:>5} logs ({pct:5.1f}%)")# Silhouette du modèle finalfinal_sil = silhouette_score(X_tfidf, df_logs['cluster_id'])print(f"\n📈 Silhouette score final: {final_sil:.3f}")

---## 6. Analyse et interprétation des clusters

In [ ]:
# Analyser le contenu de chaque clusterorder_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]top_words_per_cluster = {}print("🔍 Contenu détaillé de chaque cluster:")print("=" * 70)for cid in range(best_k):    cluster_data = df_logs[df_logs['cluster_id'] == cid]    cluster_size = len(cluster_data)    # Top termes TF-IDF    top_terms = [feature_names[idx] for idx in order_centroids[cid, :8]]    top_words_per_cluster[cid] = top_terms    # Distribution des niveaux dans ce cluster    level_dist = cluster_data['level'].value_counts().to_dict()    # Distribution des patterns    pattern_dist = cluster_data['pattern'].value_counts().head(3).to_dict()    # Déterminer la sévérité    error_ratio = level_dist.get('ERROR', 0) / cluster_size if cluster_size > 0 else 0    if error_ratio > 0.5:        severity = "🔴 CRITIQUE"    elif error_ratio > 0.2:        severity = "🟡 WARNING"    else:        severity = "🟢 NORMAL"    print(f"\n{severity} — Cluster {cid} ({cluster_size} logs, {cluster_size/len(df_logs)*100:.1f}%)")    print(f"   Top termes: {', '.join(top_terms)}")    print(f"   Niveaux: {level_dist}")    print(f"   Patterns: {pattern_dist}")    # Exemples de messages    print(f"   Exemples:")    for msg in cluster_data['message'].head(3).values:        print(f"     → {msg[:75]}{'...' if len(msg) > 75 else ''}")print("\n" + "=" * 70)

### 6.1 — Labeling opérationnel des clusters

In [ ]:
# Attribution automatique de labels basée sur les patterns dominantsdef auto_label_cluster(cluster_data, top_terms):    """Génère un label opérationnel pour un cluster."""    patterns = cluster_data['pattern'].value_counts()    dominant_pattern = patterns.index[0] if len(patterns) > 0 else 'unknown'    dominant_level = cluster_data['level'].mode().iloc[0] if len(cluster_data) > 0 else 'UNKNOWN'    # Mapping pattern → label métier    pattern_labels = {        'db_connection_error': 'Erreurs Base de Données',        'connection_timeout': 'Timeouts Connexion',        'payment_timeout': 'Erreurs Paiement',        'auth_failure': 'Échecs Authentification',        'page_visit': 'Trafic Web Normal',        'health_check': 'Health Checks',        'login_success': 'Connexions Réussies',        'oom_error': 'Erreurs Mémoire',        'ssl_error': 'Erreurs SSL/TLS',        'disk_warning': 'Alertes Disque',        'cpu_warning': 'Alertes CPU',        'backup_complete': 'Opérations Backup',        'permission_denied': 'Erreurs Permission',        'stacktrace': 'Stacktraces Applicatifs',    }    label = pattern_labels.get(dominant_pattern, f'{dominant_level} - {dominant_pattern}')    return label# Appliquer les labelscluster_labels = {}for cid in range(best_k):    cluster_data = df_logs[df_logs['cluster_id'] == cid]    label = auto_label_cluster(cluster_data, top_words_per_cluster[cid])    cluster_labels[cid] = labeldf_logs['cluster_label'] = df_logs['cluster_id'].map(cluster_labels)# Résumé labelliséprint("🏷️  Labels des clusters:")print("-" * 60)summary = df_logs.groupby(['cluster_id', 'cluster_label']).size().reset_index(name='count')summary['pct'] = (summary['count'] / len(df_logs) * 100).round(1)summary = summary.sort_values('count', ascending=False)for _, row in summary.iterrows():    print(f"   Cluster {row['cluster_id']}: {row['cluster_label']:<35} ({row['count']} logs, {row['pct']}%)")

---## 7. Visualisation des clusters

In [ ]:
# Distribution des clusters (pie + bar)fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))# Bar chartcluster_counts = df_logs['cluster_id'].value_counts().sort_index()colors = plt.cm.Set2(np.linspace(0, 1, best_k))bars = ax1.bar(cluster_counts.index, cluster_counts.values, color=colors, edgecolor='white')ax1.set_title('Distribution des clusters', fontsize=13, fontweight='bold')ax1.set_xlabel('Cluster ID')ax1.set_ylabel('Nombre de logs')for bar, label_id in zip(bars, cluster_counts.index):    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,             cluster_labels.get(label_id, '')[:20], ha='center', va='bottom', fontsize=8, rotation=15)ax1.grid(axis='y', alpha=0.3)# Pie chartax2.pie(cluster_counts.values,        labels=[f"C{i}: {cluster_labels.get(i, '')[:25]}" for i in cluster_counts.index],        colors=colors, autopct='%1.1f%%', startangle=90)ax2.set_title('Répartition des logs par cluster', fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# Heatmap: Cluster × Niveau de logct = pd.crosstab(df_logs['cluster_id'], df_logs['level'], normalize='index') * 100fig, ax = plt.subplots(figsize=(10, max(4, best_k * 0.8)))im = ax.imshow(ct.values, cmap='YlOrRd', aspect='auto')ax.set_xticks(range(len(ct.columns)))ax.set_xticklabels(ct.columns)ax.set_yticks(range(len(ct.index)))ax.set_yticklabels([f"C{i}: {cluster_labels.get(i, '')[:30]}" for i in ct.index])for i in range(len(ct.index)):    for j in range(len(ct.columns)):        ax.text(j, i, f"{ct.values[i, j]:.0f}%", ha='center', va='center', fontsize=10)plt.colorbar(im, label='% du cluster')ax.set_title('Heatmap: Distribution des niveaux par cluster', fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

---## 8. Export du résumé des clusters

In [ ]:
# Construire le résumé finalsummary_rows = []for cid in range(best_k):    cluster_data = df_logs[df_logs['cluster_id'] == cid]    count = len(cluster_data)    error_count = (cluster_data['level'] == 'ERROR').sum()    error_pct = error_count / count * 100 if count > 0 else 0    if error_pct > 50:        priority = '🔴 High'    elif error_pct > 20:        priority = '🟡 Medium'    else:        priority = '🟢 Low'    summary_rows.append({        'cluster_id': cid,        'label': cluster_labels.get(cid, 'Unknown'),        'count': count,        'percentage': round(count / len(df_logs) * 100, 1),        'top_terms': ', '.join(top_words_per_cluster[cid][:5]),        'error_rate': round(error_pct, 1),        'priority': priority,        'example': cluster_data['message'].iloc[0][:100] if count > 0 else '',    })df_summary = pd.DataFrame(summary_rows)df_summary = df_summary.sort_values('count', ascending=False)# Sauvegardersummary_path = 'data/log_clusters_summary.csv'df_summary.to_csv(summary_path, index=False)print("📋 Résumé des clusters exporté:")print(df_summary.to_string(index=False))print(f"\n💾 Sauvegardé: {summary_path}")

---## 9. Surveillance continue — Détection de nouveaux patternsCe bloc montre comment détecter si de nouveaux logs appartiennent à un cluster connu ou représentent un pattern inconnu.

In [ ]:
def classify_new_logs(new_csv_path=CSV_PATH):    """    Recharge le CSV (après un collect_logs.py) et classifie les nouveaux logs    avec le modèle K-Means entraîné.    """    # Recharger le CSV mis à jour    df_new = pd.read_csv(new_csv_path)    new_count = len(df_new) - len(df_logs)    if new_count <= 0:        print("Aucun nouveau log. Exécutez collect_logs.py pour rafraîchir.")        return    print(f"🆕 {new_count} nouveaux logs détectés!")    # Préparer les nouveaux logs    df_fresh = df_new.tail(new_count).copy()    df_fresh['message_clean'] = df_fresh['message'].astype(str).str.strip().str.lower()    df_fresh['message_normalized'] = df_fresh['message_clean'].apply(normalize_message)    df_fresh['clustering_text'] = df_fresh['level'].str.lower() + ' ' + \                                   df_fresh['pattern'] + ' ' + \                                   df_fresh['message_normalized']    # Vectoriser avec le même vectorizer    X_new = vectorizer.transform(df_fresh['clustering_text'])    # Prédire le cluster    predictions = kmeans.predict(X_new)    df_fresh['cluster_id'] = predictions    df_fresh['cluster_label'] = df_fresh['cluster_id'].map(cluster_labels)    # Résumé    print("\n📊 Classification des nouveaux logs:")    for cid in sorted(df_fresh['cluster_id'].unique()):        count = (df_fresh['cluster_id'] == cid).sum()        label = cluster_labels.get(cid, 'Unknown')        print(f"   Cluster {cid} ({label}): {count} nouveaux logs")    return df_fresh# Décommenter pour tester après un nouveau collect_logs.py:# new_results = classify_new_logs()

---## ✅ Récapitulatif du pipeline| Étape | Description | Statut ||---|---|---|| 1. Collecte | `collect_logs.py` → Loki → CSV | ✅ || 2. Nettoyage | Normalisation, placeholders | ✅ || 3. TF-IDF | Vectorisation des messages | ✅ || 4. Optimisation K | Elbow + Silhouette | ✅ || 5. K-Means | Clustering final | ✅ || 6. Interprétation | Labels opérationnels | ✅ || 7. Export | `log_clusters_summary.csv` | ✅ || 8. Continu | Classifier les nouveaux logs | ✅ |### Prochaines étapes- Automatiser `collect_logs.py` via cron (toutes les 5 min)- Ajouter des alertes quand un cluster critique grossit- Intégrer avec le modèle IsolationForest existant dans `ML/`